In [3]:
import pandas as pd

df_train = pd.read_csv('/content/drive/MyDrive/CienciaDeDatos/Viajes_train.csv')
df_test = pd.read_csv('/content/drive/MyDrive/CienciaDeDatos/Viajes_test.csv')

print(df_train.shape, df_test.shape)
df_train.head()

(3512, 9) (879, 9)


,hora,dia_semana,mes,es_fin_de_semana,hora_sin,hora_cos,es_festivo,tipo_zona_encoded,label
0,15,4,7,0,-0.707107,-7.071068e-01,0,0,85
1,6,2,4,0,1.000000,6.123234e-17,0,0,26
2,13,5,7,1,-0.258819,-9.659258e-01,0,0,27
3,21,0,8,0,-0.707107,7.071068e-01,0,0,61
4,10,3,8,0,0.500000,-8.660254e-01,0,0,40


In [4]:
# Cargar el LabelEncoder / LabelBinarizer previo
le = ('/content/drive/MyDrive/CienciaDeDatos/Viajes_lter.joblib')

# Separar variables predictoras (X) y objetivo (y)
X_train = df_train.drop(columns=['label'])
y_train = df_train['label']

X_test = df_test.drop(columns=['label'])
y_test = df_test['label']

In [5]:
import joblib
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Pipeline completo: escalado + modelo (así no tienes que preocuparte
# de escalar manualmente al hacer predicciones nuevas)
pipeline_final = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        random_state=42
    ))
])

# Entrenar con todos tus datos de entrenamiento
pipeline_final.fit(X_train, y_train)

# Guardar el pipeline completo
joblib.dump(pipeline_final, 'modelo_gradient_boosting.joblib')

print("Modelo guardado como 'modelo_gradient_boosting.joblib'")

Modelo guardado como 'modelo_gradient_boosting.joblib'


In [6]:
import joblib

modelo_cargado = joblib.load('modelo_gradient_boosting.joblib')

# Predecir con datos nuevos (mismas columnas que X_train)
predicciones = modelo_cargado.predict(X_test)

In [26]:
import joblib
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. Cargar el modelo entrenado
# ------------------------------------------------------------------
modelo = joblib.load('/content/modelo_gradient_boosting.joblib')

# ------------------------------------------------------------------
# 2. Calcular los umbrales (percentiles) a partir de tus datos reales
#    de entrenamiento. Esto define qué es "poco" o "mucha" demanda
#    según lo que históricamente pasó.
#    IMPORTANTE: y_train debe ser la misma variable que usaste al
#    entrenar el modelo (la columna 'label' de tu df_train).
# ------------------------------------------------------------------
p25 = y_train.quantile(0.25)
p50 = y_train.quantile(0.50)
p75 = y_train.quantile(0.75)

print(f"Umbrales de demanda -> P25: {p25:.1f} | P50: {p50:.1f} | P75: {p75:.1f}")


def interpretar_demanda(valor, p25, p50, p75):
    """Convierte una demanda predicha (número) en una etiqueta cualitativa."""
    if valor >= p75:
        return "Muy probable que funcione (demanda alta)"
    elif valor >= p50:
        return "Probable que funcione (demanda media-alta)"
    elif valor >= p25:
        return "Poco probable que funcione (demanda media-baja)"
    else:
        return "Improbable que funcione (demanda baja)"


# ------------------------------------------------------------------
# 3. Definir el/los viaje(s) nuevo(s) a evaluar
# ------------------------------------------------------------------
import pandas as pd

viajes_nuevos = pd.DataFrame([
    {"hora": 6, "dia_semana": 3, "mes": 11, "es_fin_de_semana": 0, "hora_sin": 1.0, "hora_cos": 0.0, "es_festivo": 0, "tipo_zona_encoded": 0},
    {"hora": 6, "dia_semana": 1, "mes": 3, "es_fin_de_semana": 0, "hora_sin": 1.0, "hora_cos": 0.0, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 23, "dia_semana": 4, "mes": 4, "es_fin_de_semana": 0, "hora_sin": -0.26, "hora_cos": 0.97, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 21, "dia_semana": 4, "mes": 2, "es_fin_de_semana": 0, "hora_sin": -0.71, "hora_cos": 0.71, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 5, "dia_semana": 1, "mes": 12, "es_fin_de_semana": 0, "hora_sin": 0.97, "hora_cos": 0.26, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 21, "dia_semana": 4, "mes": 12, "es_fin_de_semana": 0, "hora_sin": -0.71, "hora_cos": 0.71, "es_festivo": 1, "tipo_zona_encoded": 2},
    {"hora": 9, "dia_semana": 3, "mes": 12, "es_fin_de_semana": 0, "hora_sin": 0.71, "hora_cos": -0.71, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 14, "dia_semana": 5, "mes": 3, "es_fin_de_semana": 1, "hora_sin": -0.5, "hora_cos": -0.87, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 2, "dia_semana": 4, "mes": 3, "es_fin_de_semana": 0, "hora_sin": 0.5, "hora_cos": 0.87, "es_festivo": 0, "tipo_zona_encoded": 0},
    {"hora": 6, "dia_semana": 1, "mes": 4, "es_fin_de_semana": 0, "hora_sin": 1.0, "hora_cos": 0.0, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 17, "dia_semana": 1, "mes": 9, "es_fin_de_semana": 0, "hora_sin": -0.97, "hora_cos": -0.26, "es_festivo": 1, "tipo_zona_encoded": 1},
    {"hora": 19, "dia_semana": 3, "mes": 12, "es_fin_de_semana": 0, "hora_sin": -0.97, "hora_cos": 0.26, "es_festivo": 0, "tipo_zona_encoded": 0},
    {"hora": 7, "dia_semana": 6, "mes": 3, "es_fin_de_semana": 1, "hora_sin": 0.97, "hora_cos": -0.26, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 17, "dia_semana": 3, "mes": 2, "es_fin_de_semana": 0, "hora_sin": -0.97, "hora_cos": -0.26, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 9, "dia_semana": 3, "mes": 6, "es_fin_de_semana": 0, "hora_sin": 0.71, "hora_cos": -0.71, "es_festivo": 1, "tipo_zona_encoded": 1},
    {"hora": 11, "dia_semana": 1, "mes": 10, "es_fin_de_semana": 0, "hora_sin": 0.26, "hora_cos": -0.97, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 15, "dia_semana": 6, "mes": 8, "es_fin_de_semana": 1, "hora_sin": -0.71, "hora_cos": -0.71, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 7, "dia_semana": 4, "mes": 2, "es_fin_de_semana": 0, "hora_sin": 0.97, "hora_cos": -0.26, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 12, "dia_semana": 0, "mes": 1, "es_fin_de_semana": 0, "hora_sin": 0.0, "hora_cos": -1.0, "es_festivo": 1, "tipo_zona_encoded": 0},
    {"hora": 23, "dia_semana": 0, "mes": 12, "es_fin_de_semana": 0, "hora_sin": -0.26, "hora_cos": 0.97, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 10, "dia_semana": 2, "mes": 1, "es_fin_de_semana": 0, "hora_sin": 0.5, "hora_cos": -0.87, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 0, "dia_semana": 2, "mes": 5, "es_fin_de_semana": 0, "hora_sin": 0.0, "hora_cos": 1.0, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 8, "dia_semana": 3, "mes": 7, "es_fin_de_semana": 0, "hora_sin": 0.87, "hora_cos": -0.5, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 1, "dia_semana": 0, "mes": 7, "es_fin_de_semana": 0, "hora_sin": 0.26, "hora_cos": 0.97, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 4, "dia_semana": 2, "mes": 12, "es_fin_de_semana": 0, "hora_sin": 0.87, "hora_cos": 0.5, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 2, "dia_semana": 0, "mes": 3, "es_fin_de_semana": 0, "hora_sin": 0.5, "hora_cos": 0.87, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 2, "dia_semana": 0, "mes": 5, "es_fin_de_semana": 0, "hora_sin": 0.5, "hora_cos": 0.87, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 13, "dia_semana": 6, "mes": 11, "es_fin_de_semana": 1, "hora_sin": -0.26, "hora_cos": -0.97, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 9, "dia_semana": 3, "mes": 3, "es_fin_de_semana": 0, "hora_sin": 0.71, "hora_cos": -0.71, "es_festivo": 1, "tipo_zona_encoded": 2},
    {"hora": 16, "dia_semana": 3, "mes": 4, "es_fin_de_semana": 0, "hora_sin": -0.87, "hora_cos": -0.5, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 6, "dia_semana": 4, "mes": 11, "es_fin_de_semana": 0, "hora_sin": 1.0, "hora_cos": 0.0, "es_festivo": 0, "tipo_zona_encoded": 0},
    {"hora": 6, "dia_semana": 2, "mes": 3, "es_fin_de_semana": 0, "hora_sin": 1.0, "hora_cos": 0.0, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 9, "dia_semana": 4, "mes": 9, "es_fin_de_semana": 0, "hora_sin": 0.71, "hora_cos": -0.71, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 11, "dia_semana": 3, "mes": 4, "es_fin_de_semana": 0, "hora_sin": 0.26, "hora_cos": -0.97, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 22, "dia_semana": 3, "mes": 9, "es_fin_de_semana": 0, "hora_sin": -0.5, "hora_cos": 0.87, "es_festivo": 0, "tipo_zona_encoded": 0},
    {"hora": 0, "dia_semana": 0, "mes": 11, "es_fin_de_semana": 0, "hora_sin": 0.0, "hora_cos": 1.0, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 12, "dia_semana": 0, "mes": 3, "es_fin_de_semana": 0, "hora_sin": 0.0, "hora_cos": -1.0, "es_festivo": 1, "tipo_zona_encoded": 3},
    {"hora": 8, "dia_semana": 5, "mes": 5, "es_fin_de_semana": 1, "hora_sin": 0.87, "hora_cos": -0.5, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 11, "dia_semana": 6, "mes": 11, "es_fin_de_semana": 1, "hora_sin": 0.26, "hora_cos": -0.97, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 23, "dia_semana": 0, "mes": 4, "es_fin_de_semana": 0, "hora_sin": -0.26, "hora_cos": 0.97, "es_festivo": 0, "tipo_zona_encoded": 0},
    {"hora": 19, "dia_semana": 3, "mes": 7, "es_fin_de_semana": 0, "hora_sin": -0.97, "hora_cos": 0.26, "es_festivo": 0, "tipo_zona_encoded": 0},
    {"hora": 4, "dia_semana": 0, "mes": 8, "es_fin_de_semana": 0, "hora_sin": 0.87, "hora_cos": 0.5, "es_festivo": 1, "tipo_zona_encoded": 0},
    {"hora": 1, "dia_semana": 1, "mes": 12, "es_fin_de_semana": 0, "hora_sin": 0.26, "hora_cos": 0.97, "es_festivo": 1, "tipo_zona_encoded": 0},
    {"hora": 0, "dia_semana": 0, "mes": 3, "es_fin_de_semana": 0, "hora_sin": 0.0, "hora_cos": 1.0, "es_festivo": 0, "tipo_zona_encoded": 0},
    {"hora": 11, "dia_semana": 1, "mes": 6, "es_fin_de_semana": 0, "hora_sin": 0.26, "hora_cos": -0.97, "es_festivo": 0, "tipo_zona_encoded": 3},
    {"hora": 22, "dia_semana": 2, "mes": 8, "es_fin_de_semana": 0, "hora_sin": -0.5, "hora_cos": 0.87, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 5, "dia_semana": 4, "mes": 4, "es_fin_de_semana": 0, "hora_sin": 0.97, "hora_cos": 0.26, "es_festivo": 0, "tipo_zona_encoded": 1},
    {"hora": 10, "dia_semana": 6, "mes": 1, "es_fin_de_semana": 1, "hora_sin": 0.5, "hora_cos": -0.87, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 5, "dia_semana": 4, "mes": 3, "es_fin_de_semana": 0, "hora_sin": 0.97, "hora_cos": 0.26, "es_festivo": 0, "tipo_zona_encoded": 2},
    {"hora": 3, "dia_semana": 2, "mes": 10, "es_fin_de_semana": 0, "hora_sin": 0.71, "hora_cos": 0.71, "es_festivo": 0, "tipo_zona_encoded": 2}
])

# ------------------------------------------------------------------
# 4. Predecir y traducir el número a una interpretación
# ------------------------------------------------------------------
predicciones_numericas = modelo.predict(viajes_nuevos)

viajes_nuevos["prediccion_valor"] = predicciones_numericas.round(2)
viajes_nuevos["interpretacion"] = [
    interpretar_demanda(v, p25, p50, p75) for v in predicciones_numericas
]

print("\nResultado de la predicción:")
print(viajes_nuevos.to_string(index=False))

Umbrales de demanda -> P25: 17.0 | P50: 37.0 | P75: 61.0

Resultado de la predicción:
 hora  dia_semana  mes  es_fin_de_semana  hora_sin  hora_cos  es_festivo  tipo_zona_encoded  prediccion_valor                                  interpretacion
    6           3   11                 0      1.00      0.00           0                  0             40.20      Probable que funcione (demanda media-alta)
    6           1    3                 0      1.00      0.00           0                  2             31.73 Poco probable que funcione (demanda media-baja)
   23           4    4                 0     -0.26      0.97           0                  2            113.31        Muy probable que funcione (demanda alta)
   21           4    2                 0     -0.71      0.71           0                  1            113.45        Muy probable que funcione (demanda alta)
    5           1   12                 0      0.97      0.26           0                  3             22.91 Poco probable 